# Data curation and molecular standardization

**Accompanies Section 4 of** *Best Practices for Unsupervised Learning in Molecular Systems* (Article v1.0).

Standardization is the step most often skipped and the one most often responsible for meaningless downstream structure. Two records that describe the same compound in different protonation or tautomeric states occupy different regions of descriptor space, and a clustering algorithm will report that difference back to you as though it were chemistry.

This notebook works with two data sets, because no single one shows the whole problem. The ZINC-250k sample arrives already cleaned, which makes it a good place to watch what a pipeline does to charge states and to a descriptor distribution, and a poor place to look for duplicates. The PROTAC sample from TPDdb holds families of molecules that differ only in stereochemistry, so it is where deduplication has something to find and where the decision about what counts as a duplicate has consequences.

### Learning objectives
- Apply a documented standardization pipeline to two real data sets and record what it changed
- Choose a deduplication key, and see why an InChIKey travels between projects better than a canonical SMILES string
- Attribute a shift in a descriptor distribution to the step that caused it, by measuring per-molecule changes instead of comparing two distributions
- Decide on chemical grounds whether stereochemistry and tautomer handling belong in your pipeline

### What this notebook is designed to make go wrong
A molecular weight shift that invites you to blame the wrong step of the pipeline, and a deduplication run that quietly merges an active degrader with its inactive enantiomer.

### What you need installed
RDKit, pandas, NumPy and matplotlib. The `environment.yml` at the repository root installs them; nothing optional.

### Roughly how long it takes
A few minutes. Standardizing the ten thousand ZINC molecules is the slow step, and the cleanup call accounts for most of it.

In [ ]:
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# One palette for every figure here. The series stay distinguishable in
# grayscale as well as in color, and marker and dash vary alongside the color,
# so nothing depends on color alone.
TEAL, PURPLE, LAVENDER, GREEN, PLUM, SLATE = (
    "#2D4F54", "#7B539E", "#B8A0D2", "#5A9448", "#9E4A78", "#3A3D4A"
)
PALETTE = [TEAL, PURPLE, LAVENDER, GREEN]


def set_style():
    """Apply the plot style used throughout these notebooks."""
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "axes.edgecolor": SLATE,
        "axes.labelcolor": SLATE,
        "axes.titlecolor": SLATE,
        "axes.linewidth": 1.0,
        "axes.grid": False,
        "xtick.color": SLATE,
        "ytick.color": SLATE,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "legend.frameon": False,
        "axes.prop_cycle": (
            mpl.cycler(color=PALETTE)
            + mpl.cycler(marker=["o", "s", "^", "D"])
            + mpl.cycler(linestyle=["-", (0, (4, 1.5)), (0, (1, 1.2)), (0, (5, 1.2, 1, 1.2))])
        ),
    })


set_style()
warnings.filterwarnings("ignore", category=FutureWarning)

# Fix a seed so the notebook reproduces. That is not the same as checking a
# conclusion survives a different seed, which we do explicitly where it matters.
SEED = 20260726
rng = np.random.default_rng(SEED)

# The example data, fetched by scripts/download_data.py.
DATA = Path.cwd().parent / "data"

## 1. Load both data sets and record where they came from

Two files, with two different jobs.

`zinc-250k-sample.csv` holds the first ten thousand records of ZINC-250k, the drug-like
purchasable subset, with logP, QED and a synthetic accessibility score copied across from the
upstream distribution. It is a large, ordinary, small-molecule library, which makes it the
right set for watching a standardization pipeline work and for measuring how far a descriptor
distribution moves as a result.

`protac-tpddb-sample.csv` holds a thousand targeted protein degraders from TPDdb, five hundred
recruiting CRBN and five hundred recruiting VHL, with the intended target and the E3 ligase
recorded for each. PROTACs are bivalent and large, and close analogues are published in
families, so this is the set where deduplication has something to find.

Write down where data came from while you still know. Provenance costs a couple of minutes at
curation time and cannot be reconstructed later. For these two files: the ZINC records come
from ZINC12/15 rather than ZINC20, with SMILES exactly as distributed and not standardized;
the PROTAC records are the TPDdb main table filtered to rows that carry a SMILES and one of
the two ligases. `scripts/download_data.py` holds the derivation for both, so you can read
what was done to each file instead of taking someone's word for it.

In [ ]:
zinc = pd.read_csv(DATA / "zinc-250k-sample.csv")
zinc["smiles"] = zinc["smiles"].str.strip()
smiles_raw = zinc["smiles"].tolist()

protac = pd.read_csv(DATA / "protac-tpddb-sample.csv")
protac["smiles"] = protac["smiles"].str.strip()

print(f"ZINC   {len(zinc):5d} records, columns {list(zinc.columns)}")
print(f"PROTAC {len(protac):5d} records, columns {list(protac.columns)}")
print(f"\nligases in the PROTAC set: {protac['ligase'].value_counts().to_dict()}")
print(f"\nfirst ZINC SMILES  : {smiles_raw[0]}")
print(f"first PROTAC SMILES: {protac['smiles'].iloc[0]}")

### See what the two data sets are before treating them as numbers

The ZINC records are ordinary drug-like molecules; the PROTACs are two to three times their size and visibly modular, a warhead and an E3-ligand joined by a linker. The difference is not cosmetic: it is why the Bemis-Murcko scaffold argument later in the guide behaves so differently for the two sets, since a scaffold that summarizes a drug-like molecule swallows almost the whole of a PROTAC.

In [ ]:
from rdkit import Chem  # imported here too: this cell runs before Section 2's import
from rdkit.Chem import Draw
from IPython.display import display


def draw_molecules(smiles_list, legends=None, n=8, per_row=4):
    """Grid depiction of the first n molecules, for orientation rather than analysis.

    Molecules that fail to parse are dropped along with their legend, so a single
    bad string does not blank the whole grid.
    """
    mols = [Chem.MolFromSmiles(s) for s in smiles_list[:n]]
    legs = list(legends[:n]) if legends is not None else None
    keep = [i for i, m in enumerate(mols) if m is not None]
    mols = [mols[i] for i in keep]
    legs = [legs[i] for i in keep] if legs is not None else None
    return Draw.MolsToGridImage(mols, molsPerRow=per_row, subImgSize=(260, 200), legends=legs)


# Two grids, because the two files are not the same kind of object.
print("ZINC drug-like molecules:")
display(draw_molecules(smiles_raw, n=8))
print("PROTAC targeted degraders:")
# Label each by the E3 ligase it recruits and the target it degrades, the two
# annotations no unsupervised method here is given.
protac_legends = [f"{lig} / {tgt}" for lig, tgt in zip(protac["ligase"], protac["target_symbol"])]
display(draw_molecules(protac["smiles"].tolist(), legends=protac_legends, n=8))

## 2. Standardize before you compare anything

Standardization brings every structure to one consistent, documented convention before any
descriptor is computed. Here is the whole sequence, written out, so that nothing in it is
hidden. Every step is a decision, and these are the ones this notebook makes:

1. **Parse and sanitize.** `Chem.MolFromSmiles` performs valence checking, aromaticity
   perception, and ring and stereochemistry perception, and returns `None` on failure. Do not
   silence the failures. A molecule that will not parse is a molecule you know nothing about,
   and how many of them there are is a property of your data set. Report it.
2. **Normalize the depiction.** `rdMolStandardize.Cleanup` applies RDKit's transformation
   list: nitro groups, N-oxides and sulfoxides are brought to a single representation, metals
   are disconnected from the organic fragment, charges are reionized in a fixed order, and
   stereochemistry is reperceived from the resulting graph. Two sources can agree about a
   molecule and still write it two ways, and without this step those two writings produce two
   different canonical strings and survive deduplication as two compounds.
3. **Select the largest fragment.** Salts, counter-ions and solvent recorded in the same field
   are almost never the object of study, and keeping them makes molecular weight and most
   whole-molecule descriptors meaningless. The fragment count is checked after cleanup, since
   disconnecting a metal can produce a second fragment that the input string did not show.
4. **Neutralize what can be neutralized.** Once a counter-ion is gone, the parent is often
   left carrying a formal charge that is an artifact of how the salt was recorded.
   `rdMolStandardize.Uncharger` adds or removes protons to clear those charges. It leaves in
   place the charges it cannot clear, a quaternary ammonium among them, and it leaves alone a
   charge paired with an opposite charge on a neighboring atom, which is how RDKit depicts
   nitro groups and N-oxides.
5. **Canonicalize, and pick a key.** The molecule-to-SMILES mapping is one-to-many, so any
   comparison, deduplication or string-based learning done on non-canonical SMILES will
   silently disagree with itself. Section 4 covers which key to compare on.
6. **Deduplicate, and decide what to do with the groups that appear.** Section 4 as well.

RDKit bundles steps 2 to 4 into `rdMolStandardize.ChargeParent`, which runs cleanup,
largest-fragment selection and uncharging in one call. Writing them out separately costs
nothing at runtime and lets you count what each one changed, which is what Section 3 does.

Neutralization is a convention for matching structures, not a claim about what exists in
solution. A compound that is charged at pH 7.4 is still charged after the uncharger has
written it as the neutral parent. If the descriptors you are about to compute depend on
ionization, logD and charge counts and anything electrostatic among them, set the protonation
state with a pKa-aware tool and record that choice instead of neutralizing and hoping.

Tautomer canonicalization and stereochemistry removal are switches, off by default, because
each is a chemical judgment rather than a technical formality. Sections 6 and 7 show what they
do to a molecule, and the closing exercise asks you to make the call on a real case.

The order of the steps matters. Fragment selection runs before uncharging, since which charges
are artifacts depends on which counter-ions have already gone. Uncharging runs before tautomer
selection, since the tautomer scoring function is defined on the neutral form. Stereochemistry
removal runs last, because it deletes information that no later step can recover.

Standardization is lossy, which is the reason to record it. The output cannot be turned back
into the input, so keep the original string in a column beside the standardized one, keep an
index back into the source file, and record the RDKit version together with the state of both
switches. Section 8 assembles that record.

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")   # parse failures are counted below, not printed

# Build the RDKit helpers once and reuse them. Constructing them per molecule
# dominates the runtime on anything larger than a few hundred structures.
LARGEST_FRAGMENT = rdMolStandardize.LargestFragmentChooser()
UNCHARGER = rdMolStandardize.Uncharger()
TAUTOMER = rdMolStandardize.TautomerEnumerator()


def standardize_mol(smiles, canonical_tautomer=False, remove_stereochemistry=False):
    """Return a standardized molecule, or None if the string cannot be used."""
    mol = Chem.MolFromSmiles(smiles)               # 1. parse and sanitize
    if mol is None:
        return None
    mol = rdMolStandardize.Cleanup(mol)            # 2. normalize the depiction
    if len(Chem.GetMolFrags(mol)) > 1:             # 3. salts and counter-ions
        mol = LARGEST_FRAGMENT.choose(mol)
    mol = UNCHARGER.uncharge(mol)                  # 4. neutralize
    if canonical_tautomer:                         # optional, see section 7
        mol = TAUTOMER.Canonicalize(mol)
    if remove_stereochemistry:                     # optional, see section 6
        Chem.RemoveStereochemistry(mol)
    return mol


def standardize(smiles, **switches):
    """Canonical SMILES for one input string, or None if it cannot be used."""
    mol = standardize_mol(smiles, **switches)
    return None if mol is None else Chem.MolToSmiles(mol)


examples = {
    "sodium acetylsalicylate, a salt": "CC(=O)Oc1ccccc1C(=O)[O-].[Na+]",
    "carnitine, a permanent cation": "C[N+](C)(C)C[C@H](O)CC(=O)[O-]",
    "dimethyl sulfoxide, written neutral": "CS(=O)C",
    "dimethyl sulfoxide, written charge-separated": "C[S+]([O-])C",
}
for label, example in examples.items():
    print(f"{label}\n   {example}\n   -> {standardize(example)}")

## 3. Measure what the pipeline changed, and report the counts that come back at zero

The counters below are properties of the data set rather than of the code, and they belong in
the methods alongside the rest of the curation description. How many structures failed to
parse, how many arrived as more than one fragment, how many were already exact duplicates of
each other, how many carried a formal charge: run the check, print the number, and report it
whatever it turns out to be. A zero you measured is a different statement from a check you
never ran.

For this file most of those counters do come back at zero. ZINC-250k is distributed already
cleaned, and the upstream file is named `250k_rndm_zinc_drugs_clean_3.csv` to say so, so
across these records there are no salts, no multi-fragment entries and no repeated SMILES
strings. Fragment selection and deduplication have nothing to find here, which is why Section
4 moves to the PROTAC set to show what they do when there is something to find.

Charge is the counter that does not come back at zero. A large minority of these records carry
a formal charge, and the uncharger acts on nearly all of them. The classifier below counts
cationic, anionic and zwitterionic records separately, and it ignores a charged atom whose
neighbor carries the opposite charge, since that pattern is a nitro group, an N-oxide or a
charge-separated sulfoxide rather than an ionization state anyone chose.

In [ ]:
mols_raw = [Chem.MolFromSmiles(s) for s in smiles_raw]
n_failed = sum(m is None for m in mols_raw)
n_fragmented = sum(m is not None and len(Chem.GetMolFrags(m)) > 1 for m in mols_raw)


def actionable_charges(mol):
    """Positive and negative formal charge that an uncharging step can act on.

    A charge whose neighbor carries the opposite charge is a depiction choice for
    a neutral functional group: a nitro group, an N-oxide, a charge-separated
    sulfoxide. Counting those as ionization states inflates the charge count and
    misattributes what the uncharger did.
    """
    positive = negative = 0
    for atom in mol.GetAtoms():
        charge = atom.GetFormalCharge()
        if charge == 0:
            continue
        if any(neighbor.GetFormalCharge() * charge < 0 for neighbor in atom.GetNeighbors()):
            continue
        if charge > 0:
            positive += charge
        else:
            negative -= charge
    return positive, negative


def charge_class(mol):
    """One of cationic, anionic, zwitterionic or neutral."""
    positive, negative = actionable_charges(mol)
    if positive and negative:
        return "zwitterionic"
    if positive:
        return "cationic"
    if negative:
        return "anionic"
    return "neutral"


zinc["charge_class"] = [charge_class(m) if m is not None else "unparsed" for m in mols_raw]

standardized = [standardize(s) for s in smiles_raw]
clean = [s for s in standardized if s is not None]

print(f"standardization: {len(smiles_raw)} records in -> "
      f"{len(set(clean))} unique structures out\n")
print(f"  failed to parse                  : {n_failed}")
print(f"  arrived as more than one fragment: {n_fragmented}")
n_charged = int((~zinc["charge_class"].isin(["neutral", "unparsed"])).sum())
print(f"  carrying a formal charge         : {n_charged}")
print(f"  duplicate raw SMILES strings     : {len(smiles_raw) - len(set(smiles_raw))}")
print(f"  duplicates after standardization : {len(clean) - len(set(clean))}")
print("\nformal charge carried by the raw records:")
print(zinc["charge_class"].value_counts().to_string())

### Watch the neutralization happen

The counts above say a large minority of these records carry a formal charge. The grids below show a few of those molecules as they arrive and after the pipeline has neutralized them, so the step is visible as a change to a structure and not only as a number in a table.

In [ ]:
# Draw the neutralization rather than describing it: a few charged records, as
# distributed and after the pipeline has run on them.
charged = [i for i, cls in enumerate(zinc["charge_class"]) if cls in ("cationic", "anionic")][:4]
before = [smiles_raw[i] for i in charged]
after = [standardize(s) for s in before]
print("charged records, as distributed:")
display(draw_molecules(before, legends=[zinc["charge_class"][i] for i in charged], n=4, per_row=4))
print("the same records after standardization (charges neutralized):")
display(draw_molecules(after, legends=["neutral"] * len(after), n=4, per_row=4))

## 4. Choose a deduplication key before you deduplicate

A canonical SMILES string makes a serviceable key inside one project and a poor one between
projects. It is canonical with respect to RDKit's algorithm, the version you ran and the flags
you passed, so a string written by a different toolkit, or by the same toolkit two releases
later, is not guaranteed to match. The usual key for matching structures across sources is the
InChIKey, a fixed-length hash of the standard InChI: fourteen characters for the skeleton, a
second block for stereochemistry and isotopes, and a final character for the protonation
state. `Chem.MolToInchiKey` returns it.

Isotopes are a third hazard in the same family: a deuterated tracer and its unlabeled parent are the same compound for most purposes but carry different InChIKeys through the isotope layer, so they survive deduplication as two records unless the isotope labels are stripped first, exactly the deliberate decision stereochemistry and tautomers demand.

That layered structure hands you two keys instead of one. Matching on the full key keeps
enantiomers apart. Matching on the first block alone treats every stereoisomer of a skeleton as
one compound, and it merges some tautomer pairs as well, since InChI resolves mobile hydrogens
in a way that canonical SMILES does not. Both keys are defensible, they answer different
questions, and the one you used goes in the methods. InChI has limits of its own: it warns or
fails outright on organometallics and on some coordination compounds, so check the value that
comes back instead of assuming there is one.

The cell below builds all three keys for the PROTAC set and counts what each collapses. The
file was assembled by dropping repeated SMILES strings, so nothing repeats at the string level,
and whatever canonicalization catches on top of that was written two ways in the source. The
skeleton key catches something else, and Section 6 and the closing exercise are about whether
catching it is what you want.

When a group does appear, the case to think about is two records that describe the same
molecule and disagree about a value. Silently keeping whichever row came first is the common
default and the worst option: it is a decision, it is unrecorded, and it is irreproducible. The
defensible choices are to aggregate (a mean or a median, if the disagreement is measurement
noise), to keep the records separate (if they describe different conditions), or to exclude
them (if the spread is too large to interpret). Look at the spread before you pick, then write
down which one you picked. Provenance pays for itself here: if every record keeps an index back
to its source row, a disagreement is diagnosable, and without one it is noise.

In [ ]:
protac_mols = [standardize_mol(s) for s in protac["smiles"]]

records = protac.copy()
records["source_row"] = np.arange(len(records))
records["canonical"] = [None if m is None else Chem.MolToSmiles(m) for m in protac_mols]
records["inchikey"] = [None if m is None else Chem.MolToInchiKey(m) for m in protac_mols]
records["skeleton"] = [None if k is None else k.split("-")[0] for k in records["inchikey"]]
records = records[records["canonical"].notna() & records["inchikey"].notna()]

print(f"{len(protac)} PROTAC records in, {len(records)} standardized\n")
for key in ("smiles", "canonical", "inchikey", "skeleton"):
    print(f"  unique {key:10s}: {records[key].nunique():5d}")


def duplicate_groups(frame, key):
    """Sizes of the groups of records that share a key, largest first."""
    sizes = frame.groupby(key).size()
    return sizes[sizes > 1].sort_values(ascending=False)


print()
for key in ("canonical", "inchikey", "skeleton"):
    groups = duplicate_groups(records, key)
    print(f"  keyed on {key:10s}: {len(groups):3d} groups of duplicates, "
          f"{int(groups.sum()) - len(groups):3d} records merged away")

# Before merging anything, check whether the records in a group agree about the
# annotations you are about to inherit from one of them.
merged = records.groupby("skeleton").agg(
    n_records=("source_row", "size"),
    targets=("target_symbol", lambda s: sorted(set(s))),
    ligases=("ligase", lambda s: sorted(set(s))),
    sources=("source", lambda s: sorted(set(s))),
)
multi = merged[merged["n_records"] > 1]
disagree = multi[(multi["targets"].map(len) > 1) | (multi["ligases"].map(len) > 1)]
print(f"\n{len(multi)} skeleton groups hold more than one record, and {len(disagree)} of them "
      f"disagree about the target or the ligase")

## 5. Attribute the shift to the step that caused it

A standardization step that changes nothing is free; one that changes a lot deserves scrutiny.
Molecular weight is the easiest descriptor to check, and the obvious way of checking it
misleads. Two overlaid histograms, one before and one after, compare two distributions and hide
the pairing: a shift of a dalton on a third of the molecules vanishes inside bins several
daltons wide, and nothing in the picture says which molecules moved or by how much. Subtract per
molecule and plot the difference.

Once the change is per molecule, its direction identifies the step responsible. There are no
salts in this file, so `LargestFragmentChooser` never fires and fragment selection cannot have
moved anything. Uncharging moves weights in both directions. Neutralizing a carboxylate adds a
proton and about 1 Da; neutralizing an ammonium removes one and about 1 Da; a doubly charged
record moves by twice as much; a quaternary ammonium does not move at all, because it has no
proton to give up. Cations outnumber anions in this sample, so most of the molecules that move
get lighter and a few get heavier. A zwitterion that carries one group of each kind gains a
proton at one end and loses one at the other, so it comes back to about where it started. The
paired plot makes all of that visible, and two overlaid histograms could not.

In [ ]:
from rdkit.Chem import Descriptors


def molecular_weight(smiles):
    """Average molecular weight, or NaN when the string cannot be parsed."""
    if smiles is None:
        return np.nan
    mol = Chem.MolFromSmiles(smiles)
    return np.nan if mol is None else Descriptors.MolWt(mol)


# Keep the two weights on the same row, so that every comparison below is paired.
# Dropping the failures from each list separately would silently misalign them.
weights = pd.DataFrame({
    "charge_class": zinc["charge_class"],
    "before": [molecular_weight(s) for s in smiles_raw],
    "after": [molecular_weight(s) for s in standardized],
})
weights["delta"] = weights["after"] - weights["before"]
paired = weights.dropna(subset=["delta"])
changed = paired[paired["delta"].abs() > 1e-6]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))

# The weight change is discrete: uncharging adds or removes whole protons, each
# about 1.008 Da, so bin on the integer proton count. A continuous histogram
# spreads those few values across bins whose edges fall at fractional masses,
# which reads as changes the data does not contain.
PROTON = 1.00794
protons = np.round(changed["delta"].to_numpy() / PROTON).astype(int)
levels = np.arange(protons.min(), protons.max() + 1)
axes[0].bar(levels, [int((protons == k).sum()) for k in levels], width=0.8, color=TEAL)
axes[0].axvline(0.0, color=SLATE, linewidth=1.0, linestyle=(0, (4, 1.5)))
axes[0].set_xticks(levels)
axes[0].set_xlabel("protons added (+) or removed (-) by uncharging, each about 1.008 Da")
axes[0].set_ylabel("molecules")
axes[0].set_title(f"{len(changed)} of {len(paired)} molecules changed weight")

classes = [c for c in ("cationic", "anionic", "zwitterionic", "neutral")
           if (paired["charge_class"] == c).any()]
means = [paired.loc[paired["charge_class"] == c, "delta"].mean() for c in classes]
counts = [int((paired["charge_class"] == c).sum()) for c in classes]
positions = np.arange(len(classes))
axes[1].barh(positions, means, color=PALETTE[:len(classes)])
axes[1].axvline(0.0, color=SLATE, linewidth=1.0)
axes[1].set_yticks(positions)
axes[1].set_yticklabels([f"{c}\n(n={n})" for c, n in zip(classes, counts)])
axes[1].invert_yaxis()
axes[1].set_xlabel("mean change in molecular weight / Da")
axes[1].set_title("Uncharging, not fragment selection, moves the weight")

fig.tight_layout()
plt.show()

print(f"records fragment selection had anything to remove from: {n_fragmented}")
print(f"molecules whose weight changed at all: {len(changed)} of {len(paired)}")
if len(changed):
    print(f"  lighter after standardization: {int((changed['delta'] < 0).sum())}")
    print(f"  heavier after standardization: {int((changed['delta'] > 0).sum())}")
    print(f"  largest decrease {changed['delta'].min():+.3f} Da, "
          f"largest increase {changed['delta'].max():+.3f} Da")
print(f"mean change over every molecule: {paired['delta'].mean():+.4f} Da")

## 6. Decide what stereochemistry is worth to your question

The fingerprint used here, and in the notebooks that follow, is ECFP4. For each atom it hashes
the environment out to a radius of two bonds, a diameter of four, which is where the 4 in the
name comes from, then folds those hashes into a fixed number of bits, commonly 1024 or 2048.
Folding makes distinct environments collide, and collisions get more frequent as molecules get
larger, so the bit length is part of the method and has to be recorded with it.

RDKit's Morgan generator ignores stereochemistry unless you ask for it. `includeChirality` is
off by default, and with it off a pair of enantiomers produces one identical bit vector. Turning
it on adds the atom chirality tags to the atom invariants. Double bond geometry is a separate
question, and the third pair below, the E and Z isomers of crotonic acid, is there so that you
can read the answer off for the generator you are running. If stereochemistry matters to your
question, choose a descriptor that can represent it and confirm on a known pair that it does.
If it does not matter, strip it explicitly, so that the data set states the decision instead of
implying it.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

pairs = [
    ("F[C@H](Cl)Br", "F[C@@H](Cl)Br"),
    ("C[C@H](N)C(=O)O", "C[C@@H](N)C(=O)O"),
    ("C/C=C/C(=O)O", "C/C=C\\C(=O)O"),
]
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=False)
chiral_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=True)

for a, b in pairs:
    ma, mb = Chem.MolFromSmiles(a), Chem.MolFromSmiles(b)
    same_plain = generator.GetFingerprint(ma) == generator.GetFingerprint(mb)
    same_chiral = chiral_generator.GetFingerprint(ma) == chiral_generator.GetFingerprint(mb)
    print(f"{a:20s} vs {b:20s}")
    print(f"   identical fingerprints, includeChirality=False: {same_plain}")
    print(f"   identical fingerprints, includeChirality=True : {same_chiral}")

## 7. Resolve tautomers deliberately, or leave them alone

Canonical SMILES do not resolve tautomers. Two records describing the same compound in
different tautomeric forms survive deduplication as distinct molecules and sit in different
regions of descriptor space. An InChIKey merges some of those pairs through its mobile hydrogen
layer and leaves others alone, so neither key disposes of the question by itself.

In [ ]:
tautomers = ["C1C=CC(=O)NC=1", "C1=CC=C(O)N=C1"]

print("canonical SMILES, no tautomer canonicalization:")
for smi in tautomers:
    print("   ", standardize(smi))

print("\nafter canonical tautomer selection:")
for smi in tautomers:
    print("   ", standardize(smi, canonical_tautomer=True))

### Check these four failure modes before you turn tautomer canonicalization on

The article states them; here they are on real molecules, so you can change the input and watch
each one happen. RDKit's `TautomerEnumerator` picks a canonical form with a heuristic score in
which aromatization outweighs any number of carbonyls, so **the canonical tautomer is very
likely not the most stable one**. All it guarantees is that the same input gives the same
output.

Four consequences follow, and all four are silent:

1. **Stereochemistry is discarded by default**, so enantiomers collapse into one record.
2. **Explicit hydrogens turn the call into a no-op**, so data of mixed provenance ends up partially canonicalized.
3. **Enumeration can truncate**, and a truncated enumeration is not canonical at all: the status flag is the only way to find out.
4. **The rules changed at RDKit 2022.03**, so "the canonical tautomer" depends on the version you ran rather than on the molecule.

The last one cannot be shown in a single session, which is why you record the version you ran.

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
import rdkit

enumerator = rdMolStandardize.TautomerEnumerator()
print(f"RDKit {rdkit.__version__}. Record this: the rules changed at 2022.03\n")


def report(label, smiles, mol=None, enum=None):
    """Canonicalize and report what survived."""
    enum = enum if enum is not None else enumerator
    mol = mol if mol is not None else Chem.MolFromSmiles(smiles)
    before = Chem.MolToSmiles(mol)
    canon = enum.Canonicalize(mol)
    after = Chem.MolToSmiles(canon)
    stereo_before = len(Chem.FindMolChiralCenters(mol, useLegacyImplementation=False))
    stereo_after = len(Chem.FindMolChiralCenters(canon, useLegacyImplementation=False))
    rings_before = rdMolDescriptors.CalcNumAromaticRings(mol)
    rings_after = rdMolDescriptors.CalcNumAromaticRings(canon)
    flag = "  <-- CHANGED" if (stereo_after < stereo_before or rings_after > rings_before) else ""
    print(f"{label}")
    print(f"   in  {before}")
    print(f"   out {after}")
    print(f"   stereocenters {stereo_before} -> {stereo_after} | "
          f"aromatic rings {rings_before} -> {rings_after}{flag}\n")


# (1) stereochemistry is discarded by default
report("1. sp3 stereocenter, default settings",
       "C[C@H](O)C(=O)Cc1ccccc1")

# ... and is kept once you ask for it
keep_stereo = rdMolStandardize.TautomerEnumerator()
keep_stereo.SetRemoveSp3Stereo(False)
report("   same molecule, SetRemoveSp3Stereo(False)",
       "C[C@H](O)C(=O)Cc1ccccc1", enum=keep_stereo)

# (2) explicit hydrogens make it a silent no-op
plain = Chem.MolFromSmiles("O=C1CCCCC1")
with_hs = Chem.AddHs(plain)
canon_plain = Chem.MolToSmiles(Chem.RemoveHs(enumerator.Canonicalize(plain)))
canon_hs = Chem.MolToSmiles(Chem.RemoveHs(enumerator.Canonicalize(with_hs)))
print("2. explicit hydrogens")
print(f"   without explicit Hs -> {canon_plain}")
print(f"   with explicit Hs    -> {canon_hs}")
print(f"   {'identical' if canon_plain == canon_hs else 'DIFFERENT: the call was a no-op on the H-bearing graph'}\n")

# (3) enumeration can truncate, and only the status says so
tight = rdMolStandardize.TautomerEnumerator()
tight.SetMaxTautomers(4)
res = tight.Enumerate(Chem.MolFromSmiles("O=C(C)Cc1ccc(O)cc1C(=O)CC(=O)C"))
print("3. truncation")
print(f"   {len(res)} tautomers returned, status = {res.status}")
print("   Anything but 'Completed' means the 'canonical' form is not canonical.\n")

# (4) aromatization outweighs carbonyls in the scoring function
report("4. scoring: aromatization wins", "O=C1C=CC(=O)C=C1")

## 8. Record what the pipeline did, because you cannot undo it

Every step above removes something: a counter-ion, a formal charge, a proton, and with the
switches on a stereocenter or a tautomeric form. The standardized structure carries no memory
of what it used to be, so the record has to be written down beside it. What follows is the
minimum: the library version, the steps in the order they ran, the state of both switches, the
counters from Section 3, and the key the deduplication used. Keep the original string and an
index into the source file in the table itself, and a reviewer can reconstruct exactly which
molecules the analysis saw.

In [ ]:
import rdkit

pipeline_record = {
    "rdkit_version": rdkit.__version__,
    "steps": ["MolFromSmiles", "rdMolStandardize.Cleanup", "LargestFragmentChooser",
              "Uncharger", "MolToSmiles"],
    "canonical_tautomer": False,
    "remove_stereochemistry": False,
    "deduplication_key": "InChIKey, full key",
    "zinc_records_in": len(zinc),
    "zinc_unique_structures_out": len(set(clean)),
    "zinc_failed_to_parse": n_failed,
    "zinc_multi_fragment": n_fragmented,
    "zinc_charge_classes": zinc["charge_class"].value_counts().to_dict(),
    "protac_records_in": len(protac),
    "protac_unique_inchikeys": int(records["inchikey"].nunique()),
}
for name, value in pipeline_record.items():
    print(f"{name:28s} {value}")

# The curated table keeps the input beside the output and an index back to the
# source file, which is what makes the loss above auditable rather than final.
curated = zinc[["smiles", "logP", "qed", "SAS", "charge_class"]].copy()
curated = curated.rename(columns={"smiles": "smiles_as_distributed"})
curated["canonical"] = standardized
curated["source_row"] = np.arange(len(curated))
print()
print(curated.head(3).to_string())

### Exercise: decide whether these duplicates are duplicates

The cell below runs the pipeline over the PROTAC set with `remove_stereochemistry=True`, and
with `canonical_tautomer` left off, then prints the groups of records that become identical as
a result. Every group collapses for the same reason: its records differ at exactly one
stereocenter. Where that center sits varies across the groups, and the printout says which
ligase each group belongs to. In the CRBN series it is often the glutarimide of the E3 ligand.
In the rest it sits in the linker or in the target-binding warhead.

Read the groups, then answer three questions in the cell below.

Which record would you keep, and on what evidence? For a CRBN degrader a pair that differs at
the glutarimide is not a pair of interchangeable records, since the (S)-configured glutarimide
is the one that binds cereblon and the (R) enantiomer is usually much weaker. A pipeline that
merges them keeps one structure and one identifier, and whichever it kept is now the structure
your analysis attributes to both records.

Would you make the same call for a set of ZINC-like screening compounds? Flattening
stereochemistry there collapses very little and costs correspondingly little, which is a
different situation rather than a different principle.

What complicates the answer for this particular stereocenter? The glutarimide center epimerizes
in solution, which is why single-enantiomer thalidomide analogues racemize under physiological
conditions. Someone will offer that as an argument for merging the records. The argument against
is that the source recorded a configuration, and deleting it without saying so is not the same
as deciding it does not matter.

Write down your decision and the reason for it, and add both to the record built in Section 8.

In [ ]:
flattened = protac.copy()
flattened["source_row"] = np.arange(len(flattened))
flattened["flat"] = [standardize(s, remove_stereochemistry=True) for s in protac["smiles"]]
flattened = flattened[flattened["flat"].notna()]

collapsed = duplicate_groups(flattened, "flat")
print(f"{flattened['flat'].nunique()} unique structures once stereochemistry is removed, "
      f"from {len(flattened)} records")
print(f"{len(collapsed)} groups collapse, merging away "
      f"{int(collapsed.sum()) - len(collapsed)} records")
in_groups = flattened[flattened["flat"].isin(collapsed.index)]
print(f"the records in those groups by ligase: "
      f"{in_groups['ligase'].value_counts().to_dict()}\n")

for key in collapsed.index[:3]:
    group = flattened[flattened["flat"] == key]
    print(f"group of {len(group)}: target {sorted(set(group['target_symbol']))}, "
          f"ligase {sorted(set(group['ligase']))}, source {sorted(set(group['source']))}")
    for _, row in group.iterrows():
        print(f"   {row['tpd_id']}  {row['smiles'][:84]}...")
    print()

# YOUR CODE HERE
# Take one group above and find the atoms where its records differ. Then decide
# whether your pipeline should merge them, and write the decision and the reason
# in the cell below.

_Your answer:_